In [1]:
import dataset as ds
from dataset import TensorToImg, ImgWrite, ImgRead, ImgToTensor
# dt = ds.Coco("/home/wanderer2414/coco2017/")
from dataset2 import YOLODataset, test
import config
from torch import tensor, arange, float as tfloat, stack
import matplotlib.pyplot as plt
import matplotlib.patches as pat
dt = ds.Coco("D:/Datasets/PASCAL_VOC")
# i = 1
# ar = dt.getTrainTensor(0)
# boxes = dt.getTrainLabel(0)
# print(boxes.shape)
# for box in boxes:
#     x1,y1,x2,y2,cls = box.detach().numpy()
#     # print(x,y,w,h,cls, config.COCO_LABELS[cls.astype(int)])
#     # x = (x-w/2)*W
#     # w = w*W
#     # y = (y-h/2)*H
#     # h = h*H
#     rect = pat.Rectangle((x1, y1), x2-x1, y2-y1, facecolor='none', edgecolor='red')
#     plt.subplot().add_patch(rect)
# # ar = ar.permute(2, 0, 1).unsqueeze(0)/255
# img = TensorToImg(ar)
# plt.imshow(img)
# label = boxes
# H, W = ar.shape[-2:]
# X1 = label[:, 0].floor().long()
# X2 = label[:, 2].ceil().long()
# Y1 = label[:, 1].floor().long()
# Y2 = label[:, 3].ceil().long()
# C_gt = label.shape[0]
# row = arange(H, device=label.device, dtype=tfloat).view(1,1,H,1).expand(1,C_gt,H,W)
# col = arange(W, device=label.device, dtype=tfloat).view(1,1,1,W).expand(1,C_gt,H,W)
# center_x = ((X1+X2)/2).view(1, -1, 1, 1).expand(1, C_gt, H, W)
# center_y = ((Y1+Y2)/2).view(1, -1, 1, 1).expand(1, C_gt, H, W)
# distance = ((col-center_x).square() + (row-center_y).square()).sqrt()
# target = distance.min(dim=1, keepdim=True).values
# target = 1-target/target.max()
# print(target.shape)
# target = target.repeat(1, 3, 1, 1)
# img = TensorToImg(target)
# plt.imshow(img)


d:\MyRCNN\config.py:65: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(
c:\Users\2015p\miniconda3\envs\workspace\lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
d:\MyRCNN\config.py:75: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
d:\MyRCNN\config.py:78: UserWarning: Argument(s) 'mode, cval' are not valid for transform Affine
  A.Affine(shear=15, p=0.5, mode=cv2.BORDER_CONSTANT, cval=0), # Updated from deprecated IAAAffine
d:\MyRCNN\config.py:97: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(


In [2]:
# import MyRCNN
# import torch
# dev = "cpu"
# model = MyRCNN.Model(device=torch.device(dev))
# model.model.load_state_dict(torch.load("bbx.pth", map_location=dev))
def count_parameters(model):
        return sum(p.numel() for p in model.parameters())

# print(count_parameters(model.model.color))

In [3]:
# import MyRCNN
# import torch
# device = "cuda"
# model = MyRCNN.Model(dt,device=torch.device(device))
# print(count_parameters(model.model))
# # for module in model.model.color.prepare.modules():
# #     print(module)
# model.train()

In [4]:
import MyRCNN
import torch
device = "cpu"
model = MyRCNN.Model(dt,device=torch.device(device))
print(count_parameters(model.model))
model = model.model

6304


In [5]:
import os
import onnx
from onnxsim import simplify
# 4. Set the model to evaluation mode before using it for inference
model.eval()

# Create a dummy tensor matching your expected input resolution
# Use image-like values in [0, 1], because ColorHead.mode_pool2d assumes pixel intensities in that range.
dummy_input = torch.rand(1, 3, config.IMAGE_SIZE, config.IMAGE_SIZE, device='cpu')

# Sanity check the forward pass before exporting
with torch.no_grad():
    outputs = model(dummy_input)
print('forward output shapes:', [o.shape for o in outputs])

# 5. Export to ONNX
raw_onnx_path = "myRCNN_cpu.onnx"
torch.onnx.export(
    model,                      # The loaded PyTorch model
    dummy_input,                # The tracer tensor
    raw_onnx_path,              # The output file name
    export_params=True,         # Store the trained weights inside the ONNX file
    opset_version=11,           # The ONNX version (11 or 12 is highly stable for YOLO models)
    do_constant_folding=True,   # Optimizes constant math operations out of the graph
    input_names=['images'],     # Name the input node
    output_names=['boundary','mask','color','score','bbx'],
    dynamic_axes={
        'images': {0: 'batch_size'},
        'boundary': {0: 'batch_size'},
        'mask': {0: 'batch_size'},
        'color': {0: 'batch_size'},
        'score': {0: 'batch_size'},
        'bbx': {0: 'batch_size'}
    }
)

print(f"Export complete: {raw_onnx_path}")

# 6. Simplify the exported ONNX model
print("Running ONNX simplifier...")
onnx_model = onnx.load(raw_onnx_path)
model_simp, check = simplify(onnx_model)

assert check, "Simplified ONNX model could not be validated!"

simplified_onnx_path = "myRCNN_cpu_simplified.onnx"
onnx.save(model_simp, simplified_onnx_path)

print(f"Simplification complete: {simplified_onnx_path}")

forward output shapes: [torch.Size([1, 1, 320, 320]), torch.Size([1, 32, 320, 320]), torch.Size([1, 32, 320, 320]), torch.Size([1, 96, 320, 320]), torch.Size([1, 32, 5])]


d:\MyRCNN\MyRCNN\ColorHead\_model_.py:82: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  n = floor(log(min(downgrade.shape[2], downgrade.shape[3]), 3))
d:\MyRCNN\MyRCNN\ColorHead\_model_.py:82: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  n = floor(log(min(downgrade.shape[2], downgrade.shape[3]), 3))
c:\Users\2015p\miniconda3\envs\workspace\lib\site-packages\torch\onnx\_internal\jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at C:\ac

Export complete: myRCNN_cpu.onnx
Running ONNX simplifier...
Simplification complete: myRCNN_cpu_simplified.onnx


In [6]:
import onnxruntime as ort
import numpy as np

# Basic ONNX sanity check before benchmarking
session = ort.InferenceSession(simplified_onnx_path, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
check_input = np.random.rand(1, 3, config.IMAGE_SIZE, config.IMAGE_SIZE).astype(np.float32)
check_outputs = session.run(None, {input_name: check_input})
print('ONNX sanity check outputs:', len(check_outputs))
print('ONNX sanity check shapes:', [out.shape for out in check_outputs])

ONNX sanity check outputs: 5
ONNX sanity check shapes: [(1, 1, 320, 320), (1, 32, 320, 320), (1, 32, 320, 320), (1, 96, 320, 320), (1, 32, 5)]


In [7]:
from torch import Tensor,cdist
def nms(boxes: Tensor, threshold : float) -> Tensor:
    cx = (boxes[:, 0] + boxes[:,2])/2
    cy = (boxes[:, 1] + boxes[:,3])/2
    N = boxes.shape[0]
    c = stack([cx, cy], dim=-1)
    a = c.view(1, N, 2).expand(N, N, 2)
    b = c.view(N, 1, 2).expand(N, N, 2)
    x = arange(N).view(1, N).expand(N, N)
    y = arange(N).view(N, 1).expand(N, N)
    indices = y > x
    d = (a - b).square().sum(dim=-1)
    d = d*indices
    print(d.shape)
    return boxes
    


In [8]:
import onnxruntime as ort
import numpy as np
import time
import config
from utils import get_loaders
from tqdm import tqdm

# 1. Load the ONNX model and specify CPU execution
session = ort.InferenceSession("myRCNN_cpu.onnx", providers=['CPUExecutionProvider'])

# Get the exact input name defined during export (e.g., 'images')
input_name = session.get_inputs()[0].name

# Create a dummy NumPy array matching your input dimensions
dummy_input = np.random.rand(1, 3, config.IMAGE_SIZE, config.IMAGE_SIZE).astype(np.float32)

train_loader, test_loader, train_eval_loader = get_loaders(
    train_csv_path=config.TRAIN_DIR, test_csv_path=config.TEST_DIR
)
# 2. Warmup Phase (Still critical for CPU cache initialization)
for _ in range(10):
    session.run(None, {input_name: dummy_input})

# 3. Benchmarking Phase
latencies = []
for (x, labels) in tqdm(test_loader, desc="Benchmarking Model"):
    x = x.to('cpu')
    img = x.cpu().numpy()
    start_time = time.time()
    session.run(None, {input_name: img})
    end_time = time.time()
    latencies.append(end_time - start_time)

# 4. Results
avg_time_seconds = np.mean(latencies)
std_dev_seconds = np.std(latencies)
fps = 1.0 / avg_time_seconds

# Convert to milliseconds for readability
print(f"Average Inference Time: {avg_time_seconds*1000:.2f} ms")
print(f"Standard Deviation: ±{std_dev_seconds*1000:.2f} ms")
print(f"Throughput: {fps:.2f} FPS")
# print(f"ONNX CPU Average Inference Time: {avg_time_seconds*1000:.2f} ms")

Benchmarking Model:   0%|          | 13/4951 [00:43<4:38:26,  3.38s/it]


KeyboardInterrupt: 